# AI Training Studio — Google Colab

เปิดลิงก์นี้บนบัญชี Colab ของตัวเอง แล้วเลือก **Runtime → Run all**
แต่ละคนได้สำเนาโปรแกรมและดิสก์ของตัวเอง ไม่ปนกับคนอื่น

ก่อนรัน: **Runtime → Change runtime type → GPU** (T4 หรือ L4)

เมื่อมีเวอร์ชันใหม่บน GitHub ให้ **Runtime → Restart session** แล้ว Run all อีกครั้ง โค้ดจะถูกแทนที่ด้วยเวอร์ชันล่าสุด ข้อมูลใน `/content/ai_training_workspace` ยังอยู่จนกว่า runtime จะถูกตัด

ดิสก์ Colab เป็นพื้นที่ชั่วคราว ถ้า runtime ถูกยุติ ชุดข้อมูลและ checkpoint หาย ดาวน์โหลด `best.pt` จากหน้า Training หรือ Models ก่อนปิด

In [ ]:
# 1. ตรวจสอบ GPU และสภาพแวดล้อมของ Google Colab
import os, platform, subprocess, sys

print('Python:', platform.python_version())
res = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if res.returncode == 0:
    print(res.stdout)
else:
    print('⚠️ ไม่พบ GPU — แนะนำไปที่ Runtime -> Change runtime type -> เลือก T4 / L4 GPU')

In [ ]:
# 2. ดึงโค้ดล่าสุดจาก GitHub มาไว้บน runtime นี้
import os
import shutil
from pathlib import Path

REPO_URL = 'https://github.com/Atiruj2911/AI_Colab.git'
PROJECT_DIR = Path('/content/AI_Colab')
WORKSPACE_DIR = Path('/content/ai_training_workspace')

if (PROJECT_DIR / '.git').exists():
    !git -C "{PROJECT_DIR}" fetch origin
    !git -C "{PROJECT_DIR}" reset --hard origin/main
else:
    if PROJECT_DIR.exists():
        shutil.rmtree(PROJECT_DIR)
    !git clone {REPO_URL} {PROJECT_DIR}

assert (PROJECT_DIR / 'app.py').exists(), f'ไม่พบ app.py ที่ {PROJECT_DIR}'
os.chdir(PROJECT_DIR)
os.environ['AI_STUDIO_WORKSPACE'] = str(WORKSPACE_DIR)
print('Project Dir  :', PROJECT_DIR)
print('Workspace Dir:', WORKSPACE_DIR)

In [ ]:
# 3. ติดตั้ง Dependencies ที่กำหนดเวอร์ชันเข้ากันได้ (ไม่ทับ PyTorch CUDA ของ Colab)
!pip install -q -r requirements.txt

# พิมพ์เวอร์ชันแพ็กเกจทั้งหมดก่อนเปิดแอป
from pathlib import Path
from infrastructure.monitoring.system_monitor import format_startup_banner
print(format_startup_banner(Path(os.environ['AI_STUDIO_WORKSPACE'])))

In [ ]:
# 4. เปิด AI Training Studio (Gradio Web UI)
# Training จะรันใน Background Subprocess แยกจาก UI เพื่อให้ Reconnect ได้หากหน้าเว็บหลุด
!python app.py --share --workspace /content/ai_training_workspace